# Anomaly Detection

Identifying abnormal volatility events using z-scores.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load the dataset
df = pd.read_csv('../data/processed/analytics_metrics.csv', parse_dates=['date'])

print("Dataset shape:", df.shape)
print("\nanomaly_flag value counts:")
print(df['anomaly_flag'].value_counts())

## 1. Volatility Z-Score Over Time

In [ ]:
plt.rcParams['figure.figsize'] = (14, 5)

fig, ax = plt.subplots()

for ticker, group in df.groupby('ticker'):
    group_sorted = group.sort_values('date')
    ax.plot(group_sorted['date'], group_sorted['volatility_z_score'], label=ticker, linewidth=1)

# Anomaly threshold lines
ax.axhline(y=2,  color='red',  linestyle='--', linewidth=1, label='Threshold +2')
ax.axhline(y=-2, color='blue', linestyle='--', linewidth=1, label='Threshold -2')

ax.set_title('Volatility Z-Score Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility Z-Score')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45)
ax.legend(loc='upper left', fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Anomaly Events Highlighted on Volatility Chart

In [ ]:
spy = df[df['ticker'] == 'SPY'].sort_values('date')
spy_anomalies = spy[spy['anomaly_flag'] == 1]

fig, ax = plt.subplots()

# Rolling volatility line
ax.plot(spy['date'], spy['rolling_vol_30'], color='steelblue', linewidth=1.5, label='Rolling Vol (30d)')

# Anomaly flag overlay
ax.scatter(spy_anomalies['date'], spy_anomalies['rolling_vol_30'],
           color='red', s=30, zorder=5, label='Anomaly (flag=1)')

ax.set_title('SPY — Rolling Volatility with Anomaly Flags')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Volatility (30-day)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Anomaly Count by Ticker

In [ ]:
anomaly_counts = (
    df[df['anomaly_flag'] == 1]
    .groupby('ticker')
    .size()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots()

ax.bar(anomaly_counts.index, anomaly_counts.values, color='tomato', edgecolor='white')

ax.set_title('Anomaly Count by Ticker')
ax.set_xlabel('Ticker')
ax.set_ylabel('Number of Anomaly Events')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Anomaly Events Table (Most Recent 20)

In [ ]:
cols = ['date', 'ticker', 'daily_return', 'rolling_vol_30', 'volatility_z_score']

recent_anomalies = (
    df[df['anomaly_flag'] == 1]
    .sort_values('date', ascending=False)
    .head(20)
    [cols]
    .reset_index(drop=True)
)

recent_anomalies

## 5. Z-Score Distribution

In [ ]:
z_scores = df['volatility_z_score'].dropna()

fig, ax = plt.subplots()

ax.hist(z_scores, bins=60, color='steelblue', edgecolor='white', alpha=0.85)

# Threshold lines
ax.axvline(x=2,  color='red',  linestyle='--', linewidth=1.5, label='+2 threshold')
ax.axvline(x=-2, color='blue', linestyle='--', linewidth=1.5, label='-2 threshold')

ax.set_title('Volatility Z-Score Distribution')
ax.set_xlabel('Volatility Z-Score')
ax.set_ylabel('Frequency')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()